# `POGGGraphConverter` usage examples

This page contains examples for using the `POGGGraphConverter` class.

[See full module documentation here](project:/apidocs/pogg/pogg.graph_to_SEMENT.graph_to_SEMENT.md)

A `POGGGraphConverter` object contains instance methods for converting graphs to SEMENTs.

In typical usage of POGG, creating and interacting with this object directly will not be necessary. Rather, one of these objects is an instance attribute of a [POGG](project:/apidocs/pogg/pogg.pogg_routine.md#pogg.pogg_routine.POGG) object.

But for those interested, this guide will show examples of all functions of the `POGGGraphConverter` object for reference.

First, we will need the following imports:

In [11]:
import os
import json
import pogg.my_delphin.sementcodecs as sementcodecs
from pogg.data_handling.graph_util import POGGGraphUtil
from pogg.pogg_routine import POGG

Let's also assume there is a `dataset.yml` file with the following information:

```yaml
# top level directory
data_dir: "./data"

# subdirectories
data_chunk: "BitsyBakery"
graph_json_dir: "graph_jsons"
graph_dot_dir: "dot"
lexicon_dir: "lexicon"
evaluation_dir: "evaluation"
gold_outputs_dir: "gold_outputs"

# other data information
dataset_name: "BitsyBakery"
```

We can use these files to initialize a `POGG` object:

In [12]:
pogg_obj = POGG("../config.yml", "../dataset.yml")

The [POGG](project:/apidocs/pogg/pogg.pogg_routine.md#pogg.pogg_routine.POGG) object contains a number of objects of the following types as instance attributes:

- [POGGConfig](project:/apidocs/pogg/pogg.pogg_config.md#pogg.pogg_config.POGGConfig)
- [POGGDataset](project:/apidocs/pogg/pogg.data_handling.pogg_dataset.md#pogg.data_handling.pogg_dataset.POGGDataset)
- [POGGEvaluation](project:/apidocs/pogg/pogg.evaluation.evaluation.md#pogg.evaluation.evaluation.POGGEvaluation)
- [SemanticAlgebra](project:/apidocs/pogg/pogg.semantic_composition.semantic_algebra.md#pogg.semantic_composition.semantic_algebra.SemanticAlgebra)
- [SemanticComposition](project:/apidocs/pogg/pogg.semantic_composition.semantic_composition.md#pogg.semantic_composition.semantic_composition.SemanticComposition)
- [POGGGraphConverter](project:/apidocs/pogg/pogg.graph_to_SEMENT.graph_to_SEMENT.md#pogg.graph_to_SEMENT.graph_to_SEMENT.POGGGraphConverter)

This guide walks through examples of instance methods of the `pogg_obj.graph_converter`object.

## `get_SEMENT` example

:::{code} Method details
:collapsible:
````{py:method} get_SEMENT(comp_fxn_name, parameter_values)
:canonical: pogg.graph_to_SEMENT.graph_to_SEMENT.POGGGraphConverter.get_SEMENT

```{autodoc2-docstring} pogg.graph_to_SEMENT.graph_to_SEMENT.POGGGraphConverter.get_SEMENT
```

````
:::

The purpose of this method is to perform a function call on the `SemanticComposition` object with the given information. For example, if we know we want to call the `noun` function from the `SemanticComposition` class with the predicate label `_cake_n_1` and the intrinsic variable property `{'NUM': 'sg'}`, we can pass this information `get_SEMENT` which will perform the call.

In [13]:
cake = pogg_obj.graph_converter.get_SEMENT("noun", {"predicate": "_cake_n_1", "intrinsic_variable_properties": {'NUM': 'sg'}})

# print the SEMENT as a string
print(sementcodecs.encode(cake, indent=True))

[ TOP: h2
  INDEX: x1 [ x NUM: sg ]
  RELS: < [ _cake_n_1 LBL: h2 ARG0: x1 ] > ]


## `convert_node_to_SEMENT` example

:::{code} Method details
:collapsible:
````{py:method} convert_node_to_SEMENT(node, node_evaluation=None)
:canonical: pogg.graph_to_SEMENT.graph_to_SEMENT.POGGGraphConverter.convert_node_to_SEMENT

```{autodoc2-docstring} pogg.graph_to_SEMENT.graph_to_SEMENT.POGGGraphConverter.convert_node_to_SEMENT
```

````
:::

The method generates a SEMENT based on given node information from a directed graph.

Imagine you have a node in the graph called `cake1` with node properties specifying the lexicon key (e.g. `{'lexicon_key': 'cake'}`) then a SEMENT can be generated from this.

:::{assumptions} Assumptions
:collapsible:
Note that for this example it is assumed that there is a lexicon with a key for `'cake'` like the following:

```json
"cake": {
    "comp_fxn": "noun",
    "predicate": "_cake_n_1",
    "intrinsic_variable_properties": {}
}
```

Also for this example we are foregoing the optional evaluation object.
:::

In [14]:
node = ("cake1", {"lexicon_key": "cake"})
cake = pogg_obj.graph_converter.convert_node_to_SEMENT(node)

# print the SEMENT as a string
print(sementcodecs.encode(cake, indent=True))

[ TOP: h4
  INDEX: x3
  RELS: < [ _cake_n_1 LBL: h4 ARG0: x3 ] > ]


## `convert_edge_to_SEMENT` example

:::{code} Method details
:collapsible:
````{py:method} convert_edge_to_SEMENT(edge, parent, child, edge_evaluation=None)
:canonical: pogg.graph_to_SEMENT.graph_to_SEMENT.POGGGraphConverter.convert_edge_to_SEMENT

```{autodoc2-docstring} pogg.graph_to_SEMENT.graph_to_SEMENT.POGGGraphConverter.convert_edge_to_SEMENT
```

````
:::

The method generates a SEMENT based on given edge information from a directed graph. In order to do this the function requires information about the edge itself, namely the properties, and the SEMENTs generated (if any) from the parent node and child node. Edges typically specify how two SEMENTs combine with one another to compose a new SEMENT. This function shouldn't be called directly by a user, but for illustrative purposes an example is below.

Imagine there's an edge, `'flavor'`, pointing from a parent node, `'cake1'` to a child node, `'vanilla'`. The edge has the following properties dictionary: `{'lexicon_key': 'flavor'}`.

:::{assumptions} Assumptions
:collapsible:
Note that for this example it is assumed that there is a lexicon with a key for `'flavor'` like the following:

```json
"flavor": {
    "comp_fxn": "compound_noun",
    "head_noun_sement": "parent",
    "non_head_noun_sement": "child"
}
```

Also for this example we are foregoing the optional evaluation object.
:::



In [15]:
# make 'vanilla' and 'cake' SEMENTs directly for the example
vanilla = pogg_obj.semantic_composition.noun("_vanilla_n_1")
cake = pogg_obj.semantic_composition.noun("_cake_n_1")

edge_info = {'lexicon_key': 'flavor'}

# SEMENT resulting from parent node first
vanilla_cake = pogg_obj.graph_converter.convert_edge_to_SEMENT(edge_info, cake, vanilla)

# print SEMENT as string
print(sementcodecs.encode(vanilla_cake, indent=True))


[ TOP: h8
  INDEX: x7
  RELS: < [ _cake_n_1 LBL: h8 ARG0: x7 ] > ]


## `convert_graph_to_SEMENT` example

:::{code} Method details
:collapsible:
````{py:method} convert_graph_to_SEMENT(root, graph, graph_evaluation=None)
:canonical: pogg.graph_to_SEMENT.graph_to_SEMENT.POGGGraphConverter.convert_graph_to_SEMENT

```{autodoc2-docstring} pogg.graph_to_SEMENT.graph_to_SEMENT.POGGGraphConverter.convert_graph_to_SEMENT
```

````
:::

The method generates a SEMENT from a provided graph. The function requires the `graph` as the first argument. It also optionally accepts a `graph_evaluation` object and a `root`.

:::{info} About the `root` argument
:collapsible:
Note that if no root is passed in the function will call the `find_root` function from the [POGGGraphUtil class](project:/apidocs/pogg/pogg.data_handling.graph_util.md). When calling the function for the first time on a graph, passing in the root isn't really expected or necessary, the main purpose of this argument is to make recursive calls simpler by specifying the root of the subgraph the recursive call is supposed to work with.
:::

Imagine there's a simple graph with two nodes, `cake` and `vanilla` with a `flavor` edge pointing from `cake` to `vanilla`.

:::{assumptions} Assumptions
:collapsible:
Note that for this example it is assumed that there is a lexicon with the following information:

```json
{
    "node_keys": {
        "cake": {
            "comp_fxn": "noun",
            "predicate": "_cake_n_1",
            "intrinsic_variable_properties": {}
        },
        "vanilla": {
            "comp_fxn": "noun",
            "predicate": "_vanilla_n_1",
            "intrinsic_variable_properties": {}
        }
    },
    "edge_keys": {
        "flavor": {
            "comp_fxn": "compound_noun",
            "head_noun_sement": "parent",
            "non_head_noun_sement": "child"
        }
    }
}
```

Also for this example we are foregoing the optional evaluation object.
:::

In [16]:
# read graph from JSON file
with open(os.path.join(pogg_obj.dataset.graph_json_dir, "vanilla_cake.json"), "r") as f:
    graph_json = json.load(f)

# build NetworkX graph from JSON information
graph = POGGGraphUtil.build_graph(graph_json)

# convert graph to SEMENT
graph_sement = pogg_obj.graph_converter.convert_graph_to_SEMENT(graph)

# print SEMENT as string
print(sementcodecs.encode(graph_sement, indent=True))

[ TOP: h10
  INDEX: x9
  RELS: < [ _cake_n_1 LBL: h10 ARG0: x9 ] > ]
